# Reporte de notas desde Excel + Exportacion a excel

- leer una hoja de excel con calificaciones, calcular estadisticas por estudiante y por actividad, detectar estudiantes en riesgo, y guardar un reporte resumido en otro excel 

In [10]:
## 1) importar librerias
# openpyxl nos permite leer y escribir archivos de Excel (.xlsx)
import openpyxl
from openpyxl import Workbook

#patch es una libreria que nos permite trabajar con rutas de archivos de manera mas sencilla
from pathlib import Path

#Definimos la carpeta de trabajo, es decir, donde se encuentran los archivos de Excel que queremos procesar
carpeta = Path('.')

#definimos el archivo de entrada, es decir, el archivo de Excel que contiene los datos que queremos procesar
archivo_entrada = carpeta / 'ejemplo_notas.xlsx'

archivo_salida = carpeta / 'ejemplo_notas_procesado.xlsx'

# mostramos el archivo de entrada para verificar que la ruta es correcta
print(f'Archivo de entrada: {archivo_entrada.resolve()}')

print(f'Archivo de salida: {archivo_salida.resolve()}')

Archivo de entrada: C:\Users\salas\AppData\Local\Programs\Microsoft VS Code\ejemplo_notas.xlsx
Archivo de salida: C:\Users\salas\AppData\Local\Programs\Microsoft VS Code\ejemplo_notas_procesado.xlsx


In [ ]:
## 2) leer el archivo de Excel y lo convertimos a estructuras de python
# abrimos el archivo de Excel utilizando openpyxl
wb = openpyxl.load_workbook(archivo_entrada)

# seleccionamos la hoja de trabajo que contiene los datos, en este caso, la hoja llamada 'Notas'
hoja = wb['Notas']

#leemos los encabezados de la hoja de trabajo para verificar que los datos están organizados correctamente
encabezados = [celda.value for celda in hoja[1]]  # la primera fila (fila 1) contiene los encabezados

print(f'Encabezados: {encabezados}')

# vamos a separar los nombres --> calumna 1 y las notas --> columnas 2 - 6

nombres = []
matriz_notas = []  # matriz =  lista de filas, cada fila = lista de numeros

# recorremos desde la segunda fila (fila 2) hasta la última fila con datos
for fila in hoja.iter_rows(min_row=2, values_only=True):
    
    nombres.append(fila[0])  # el primer valor de cada fila es el nombre del estudiante
    
    notas_E = [] # lista para almacenar las notas de cada estudiante
    
    for valor in fila[1:]:  # recorremos los valores desde la segunda columna en adelante
        if valor is not None:  # verificamos que el valor no sea None (celda vacía)
            notas_E.append(valor)  # agregamos el valor a la lista de notas_E
            
    matriz_notas.append(notas_E)  # agregamos la lista de notas_E a la matriz de notas
  
print(f'encabezados: {encabezados}') # mostramos los encabezados para verificar que se han leído correctamente
print(f'Total de estudiantes: {len(nombres)}') # el número de filas en la matriz de notas es el número de estudiantes
print(f'Total actividades: {len(matriz_notas[0])}')  # el número de columnas en la primera fila de notas es el número de actividades


FileNotFoundError: [Errno 2] No such file or directory: 'ejemplo_notas.xlsx'

In [ ]:
# 3) creamos funciones propias

def sumar_lista(lista):
    """Función que recibe una lista de números y devuelve la suma de sus elementos."""
    total = 0
    for numero in lista:
        total += numero
    return total

def promedio_lista(lista):
    """ Busca el promedio de una lista de números, es decir, la suma de los elementos dividida por la cantidad de elementos."""
    suma = sumar_lista(lista)
    cantidad = len(lista)
    if cantidad == 0:
        return 0
    return suma / cantidad

def maximo_lista(lista):
    """Busca el valor máximo en una lista de números."""
    if not lista:
        return None  # si la lista está vacía, devolvemos None
    maximo = lista[0]  # asumimos que el primer elemento es el máximo inicialmente
    for numero in lista:
        if numero > maximo:
            maximo = numero  # actualizamos el máximo si encontramos un número mayor
    return maximo

def minimo_lista(lista):
    """Busca el valor mínimo en una lista de números."""
    if not lista:
        return None  # si la lista está vacía, devolvemos None
    minimo = lista[0]  # asumimos que el primer elemento es el mínimo inicialmente
    for numero in lista:
        if numero < minimo:
            minimo = numero  # actualizamos el mínimo si encontramos un número menor
    return minimo

def posicion_de_valor(lista, valor):
    """Busca la posición de un valor específico en una lista. Devuelve la posición (índice) si se encuentra, o -1 si no se encuentra."""
    for indice in range(len(lista)):
        if lista[indice] == valor:
            return indice  # devolvemos el índice si encontramos el valor
    return -1  # devolvemos -1 si no encontramos el valor

In [ ]:
## 4) estadisticas por estudiante
# definimos un umbral para deficur si esta en 'riesgo'

umbral_riesgo = 3.0  # por ejemplo, si el promedio es menor a 3.0, consideramos que el estudiante está en riesgo

reporte_riesgo = []  # lista para almacenar el reporte de riesgo de cada estudiante

for i in range(len(nombres)):
    nombre_estudiante = nombres[i]
    notas_estudiante = matriz_notas[i]
    
    promedio_estudiante = promedio_lista(notas_estudiante) # calculamos el promedio de las notas del estudiante utilizando la función que definimos anteriormente
    maximo_estudiante = maximo_lista(notas_estudiante)  # calculamos el valor máximo de las notas del estudiante utilizando la función que definimos anteriormente
    minimo_estudiante = minimo_lista(notas_estudiante)  # calculamos el valor mínimo de las notas del estudiante utilizando la función que definimos anteriormente
    
    posicion_maximo = posicion_de_valor(notas_estudiante, maximo_estudiante)    # calculamos la posición del valor máximo en la lista de notas del estudiante utilizando la función que definimos anteriormente
    posicion_minimo = posicion_de_valor(notas_estudiante, minimo_estudiante)    # calculamos la posición del valor mínimo en la lista de notas del estudiante utilizando la función que definimos anteriormente
    
    riesgo_estudiante = promedio_estudiante < umbral_riesgo  # determinamos si el estudiante está en riesgo comparando su promedio con el umbral definido
    
    registro = {
        'estudiante': nombre_estudiante,    # agregamos el nombre del estudiante al registro
        'promedio': promedio_estudiante,    # agregamos el promedio al registro
        'maximo': maximo_estudiante,    # agregamos el valor máximo al registro
        'actividad_maximo': encabezados[posicion_maximo + 1],   # sumamos 1 al índice para obtener el nombre de la actividad correspondiente al valor máximo
        'minimo': minimo_estudiante,    # agregamos el valor mínimo al registro
        'actividad_minimo': encabezados[posicion_minimo + 1],   # sumamos 1 al índice para obtener el nombre de la actividad correspondiente al valor mínimo

        'riesgo': 'si' if riesgo_estudiante else 'no'  # agregamos la información de riesgo al registro, indicando 'si' si el estudiante está en riesgo y 'no' si no lo está
    }
    
    reporte_riesgo.append(registro)  # agregamos el registro al reporte de riesgo
    
# mostramos las primeras 5 entradas del reporte de riesgo para verificar que se ha generado correctamente
print('Reporte de riesgo (primeras 5 entradas):')
for registro in reporte_riesgo[:5]:
    print(registro)

In [ ]:
# 5) estadisticas por actividad
# vamos a calcular el promedio, máximo y mínimo de cada actividad, así como la cantidad de estudiantes en riesgo para cada actividad

reporte_actividades = []  # lista para almacenar el reporte de cada actividad

# cantidad de actividades es el número de columnas en la matriz de notas
cantidad_actividades = len(matriz_notas[0])  # asumimos que todas

# ahora recorremos cada actividad (columna) para calcular las estadísticas correspondientes
for j in range(cantidad_actividades):
    columna = []
    
    # vamos a construir la columna j recorriendo cada fila de la matriz de notas
    for i in range(len(matriz_notas)):
        columna.append(matriz_notas[i][j])  # agregamos el valor de la actividad j para el estudiante i a la columna
        
    promedio = promedio_lista(columna)  # calculamos el promedio de la columna utilizando la función que definimos anteriormente
    maximo = maximo_lista(columna)  # calculamos el valor máximo de la columna utilizando la función que definimos anteriormente
    minimo = minimo_lista(columna)  # calculamos el valor mínimo de la columna utilizando la función que definimos anteriormente
    
    reporte_actividades.append({
        'actividad': encabezados[j + 1],  # sumamos 1 al índice
        'promedio': round(promedio, 2),  # agregamos el promedio al reporte de actividades
        'maximo': maximo,  # agregamos el valor máximo al reporte de actividades
        'minimo': minimo  # agregamos el valor mínimo al reporte de actividades
    })
    
    for registro in reporte_actividades:
        print(registro) 

In [ ]:
## 6) generar un nuevo archivo de Excel con los reportes generados
# creamos un nuevo libro de Excel utilizando openpyxl

nuevo_libro = Workbook()

# hoja 1:  reporte por estudiante
hoja_estudiantes = nuevo_libro.active  # obtenemos la hoja activa del nuevo libro, que será la hoja donde escribiremos el reporte por estudiante
hoja_estudiantes.title = 'Reporte por Estudiante'  # renombramos la hoja

# escribimos los encabezados en la primera fila de la hoja de reporte por estudiante
encabezados_estudiantes = ['Estudiante', 'Promedio', 'Máximo', 'Actividad Máximo', 'Mínimo', 'Actividad Mínimo', 'Riesgo']
hoja_estudiantes.append(encabezados_estudiantes)  # agregamos los encabezados a la hoja

#ahora escribimos las filas de datos del reporte por estudiante
for registro in reporte_riesgo:
    fila = [
        registro['estudiante'],  # agregamos el nombre del estudiante a la fila
        registro['promedio'],  # agregamos el promedio a la fila
        registro['maximo'],  # agregamos el valor máximo a la fila
        registro['actividad_maximo'],  # agregamos el nombre de la actividad correspondiente al valor máximo a la fila
        registro['minimo'],  # agregamos el valor mínimo a la fila
        registro['actividad_minimo'],  # agregamos el nombre de la actividad correspondiente al valor mínimo a la fila
        registro['riesgo']  # agregamos la información de riesgo a la fila
    ]
    hoja_estudiantes.append(fila)  # agregamos la fila a la hoja


# hoja 2: reporte por actividad
hoja_actividades = nuevo_libro.create_sheet(title='Reporte por Actividad')  # creamos una nueva hoja en el libro para el reporte por actividad y la nombramos
hoja_actividades.append(['Actividad', 'Promedio', 'Máximo', 'Mínimo'])  # escribimos los encabezados en la primera fila de la hoja de reporte por actividad

for registro in reporte_actividades:
    fila = [
        registro['actividad'],  # agregamos el nombre de la actividad a la fila
        registro['promedio'],  # agregamos el promedio a la fila
        registro['maximo'],  # agregamos el valor máximo a la fila
        registro['minimo']  # agregamos el valor mínimo a la fila
    ]
    hoja_actividades.append(fila)  # agregamos la fila a la hoja
    
# guardamos el nuevo libro de Excel con un nombre específico
nuevo_libro.save(archivo_salida)  # guardamos el nuevo libro de Excel en la ruta especificada
print(f'Archivo de salida: {archivo_salida.resolve()}')  # mostramos la ruta del archivo de salida para verificar que se ha guardado correctamente